In [ ]:
# ISYS2001 Assessment 2: Student Budget and Savings Assistant

## Project Overview

My project is a Student Budget and Savings Assistant. It is designed to help university students understand their spending and manage their money.

## Step 1: Understand the Problem

### The Problem
University students may find it difficult to manage their money because they have different expenses and may not know how much money they have left after spending.

### Target User
The target users are university students who want to understand their spending habits and manage their budgets.

### Proposed Solution
I plan to build an application that analyses financial information, calculates total expenses and remaining money, and summarises spending by category. A conversational AI assistant will help users understand their results and provide relevant budgeting guidance.

### Why It Is Useful
The application can help students identify where their money goes, understand their spending habits and plan towards savings goals.

In [ ]:
## Step 2: Identify Inputs and Outputs

### Inputs
The application will receive:
- The user's monthly income.
- Financial transactions, including amounts and spending categories such as rent, groceries, transport and shopping.
- An optional savings goal.
- Questions typed by the user into the AI chat assistant.

The user can provide transaction data through a CSV file or enter relevant figures through the interface.

### Processing
The program will calculate total expenses, group expenses by category, calculate the remaining balance and, when a savings goal is provided, calculate savings goal progress.

### Outputs
The application will display:
- Total expenses.
- Money remaining after expenses.
- A breakdown of expenses by category.
- Progress towards a savings goal, when applicable.
- Responses from the AI assistant based on the user's financial data and questions.

### Example
If the user enters a monthly income of $1,500 and expenses totalling $1,000, the expected remaining balance is $500.

### Expected Behaviour
The results should change when the user provides different financial information. Invalid or incomplete inputs should be handled sensibly.

In [ ]:
## Step 3: Work an Example by Hand

I will work through an example manually to determine the expected results before writing Python code.

### Example Inputs
- Monthly income: $1,800
- Rent: $700
- Groceries: $250
- Transport: $100
- Shopping: $150

### Calculation 1: Total Expenses
Total expenses = Rent + Groceries + Transport + Shopping

Total expenses = $700 + $250 + $100 + $150

Total expenses = $1,200

### Calculation 2: Remaining Money
Remaining money = Monthly income - Total expenses

Remaining money = $1,800 - $1,200

Remaining money = $600

### Expected Outputs
- Total expenses: $1,200
- Remaining money: $600
- Highest spending category: Rent ($700)

### Conclusion
The manual calculations show that a student earning $1,800 and spending $1,200 has $600 remaining. These expected results will be used to check whether the Python program works correctly.

In [ ]:
## Step 4: Write Pseudocode

Pseudocode describes the program's logic in plain language before it is converted into Python.

```text
START

GET the user's monthly income
GET the user's financial transactions
SET total expenses to zero

FOR each transaction:
    GET the transaction amount
    ADD the amount to total expenses
    GROUP the transaction amount by spending category

CALCULATE remaining money = income - total expenses
FIND the category with the highest total spending

DISPLAY total expenses
DISPLAY remaining money
DISPLAY spending by category
DISPLAY the highest spending category

IF the user has entered a savings goal:
    CALCULATE and DISPLAY savings goal progress

END
```

### Explanation
The program collects financial information, adds the expenses together, calculates the remaining balance and summarises spending by category. If a savings goal is provided, it also calculates the relevant progress. The outputs will be used to help the user understand their budget.

In [ ]:
def calculate_remaining(income, expenses):
    total_expenses = sum(expenses)
    remaining = income - total_expenses
    return remaining

result = calculate_remaining(1800, [700, 250, 100, 150])
print("Money remaining: $", result)

Money remaining: $ 600


In [10]:

def analyse_spending(transactions):
    category_totals = {}

    try:
        for transaction in transactions:
            category = transaction["category"]
            amount = transaction["amount"]

            if not isinstance(amount, (int, float)) or isinstance(amount, bool) or amount < 0:
                return {"error": "Expense amounts must be non-negative numbers."}

            category_totals[category] = category_totals.get(category, 0) + amount

        return category_totals

    except (KeyError, TypeError):
        return {"error": "Please provide valid transaction data."}


# Example transactions
transactions = [
    {"category": "Rent", "amount": 700},
    {"category": "Groceries", "amount": 250},
    {"category": "Transport", "amount": 100},
    {"category": "Groceries", "amount": 50},
    {"category": "Shopping", "amount": 150}
]

print(analyse_spending(transactions))


{'Rent': 700, 'Groceries': 300, 'Transport': 100, 'Shopping': 150}


In [11]:

# Test normal spending
assert analyse_spending([
    {"category": "Food", "amount": 100},
    {"category": "Food", "amount": 50}
]) == {"Food": 150}

# Test an empty transaction list
assert analyse_spending([]) == {}

# Test invalid expense amount
assert "error" in analyse_spending([
    {"category": "Food", "amount": -10}
])

# Test missing category
assert "error" in analyse_spending([
    {"amount": 50}
])

print("All spending analysis tests passed!")


All spending analysis tests passed!


In [12]:

import pandas as pd

data = {
    "category": ["Rent", "Groceries", "Transport", "Groceries", "Shopping"],
    "amount": [700, 250, 100, 50, 150]
}

df = pd.DataFrame(data)
df.to_csv("transactions.csv", index=False)

transactions_df = pd.read_csv("transactions.csv")

print(transactions_df)


    category  amount
0       Rent     700
1  Groceries     250
2  Transport     100
3  Groceries      50
4   Shopping     150


In [13]:

transactions = transactions_df.to_dict("records")

print(analyse_spending(transactions))

{'Rent': 700, 'Groceries': 300, 'Transport': 100, 'Shopping': 150}


In [14]:

def analyse_uploaded_csv(file_path):
    try:
        if file_path is None:
            return "Please upload a CSV file."

        df = pd.read_csv(file_path)

        if "category" not in df.columns or "amount" not in df.columns:
            return "CSV must contain category and amount columns."

        df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

        if df["amount"].isna().any():
            return "Please make sure every amount is a valid number."

        if (df["amount"] < 0).any():
            return "Expense amounts cannot be negative."

        if df["category"].isna().any() or (df["category"].astype(str).str.strip() == "").any():
            return "Every transaction must have a category."

        transactions = df.to_dict("records")
        result = analyse_spending(transactions)

        return str(result)

    except Exception as error:
        return "Could not read the CSV file. Please check its format."


In [15]:

print(analyse_uploaded_csv("transactions.csv"))


{'Rent': 700, 'Groceries': 300, 'Transport': 100, 'Shopping': 150}


In [16]:

import gradio as gr

app = gr.Interface(
    fn=analyse_uploaded_csv,
    inputs=gr.File(
        label="Upload your transaction CSV",
        file_types=[".csv"],
        type="filepath"
    ),
    outputs=gr.Textbox(label="Spending analysis"),
    title="Student Budget and Savings Assistant",
    description="Upload a CSV containing category and amount columns."
)

app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1948b3c932b1c04792.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
from google.colab import files
files.download("transactions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
from google.colab import userdata
import requests

api_key = userdata.get("GEMINI_API_KEY")

if api_key:
    print("API key loaded successfully!")
else:
    print("API key not found. Check your Colab Secrets settings.")

API key loaded successfully!


In [19]:
def ask_gemini(question):
    url = (
        "https://generativelanguage.googleapis.com/v1beta/"
        "models/gemini-flash-latest:generateContent"
    )

    headers = {
        "Content-Type": "application/json",
        "x-goog-api-key": api_key
    }

    data = {
        "contents": [
            {
                "parts": [
                    {"text": question}
                ]
            }
        ]
    }

    try:
        response = requests.post(
            url,
            headers=headers,
            json=data,
            timeout=30
        )

        response.raise_for_status()
        result = response.json()

        return result["candidates"][0]["content"]["parts"][0]["text"]

    except requests.exceptions.RequestException:
        return "Could not connect to Gemini. Please check your API access."

    except (KeyError, IndexError, TypeError):
        return "Gemini returned an unexpected response."


print(ask_gemini("Give me one simple tip for saving money as a university student."))

**Never buy textbooks brand new.** 

Before the semester starts, check your campus library (they often keep copies on reserve), rent digital or physical copies online, buy used from older students, or search for free, legal open-source PDF versions. This single habit can easily save you $300 to $600 each semester with very little effort.


In [21]:
def ask_finance_assistant(question):
    transactions = transactions_df.to_dict("records")
    spending_summary = analyse_spending(transactions)

    prompt = f"""
    You are a helpful Student Budget and Savings Assistant.

    Here is the user's spending data:
    {spending_summary}

    The user has asked: {question}

    Give a clear, simple answer based on their spending data.
    Help them understand their expenses and save money.
    Do not invent financial information that is not provided.
    """

    return ask_gemini(prompt)


print(ask_finance_assistant("Which category am I spending the most on?"))

You are spending the most on **Rent**, which comes out to **$700**. 

Here is how your total monthly spending of **$1,250** breaks down:
* **Rent:** $700 (56% of your total spending)
* **Groceries:** $300 (24%)
* **Shopping:** $150 (12%)
* **Transport:** $100 (8%)

### Tips to Help You Save:
* **Fixed vs. Flexible Costs:** Rent is your largest cost, but as a student, it is often a fixed expense that is difficult to change immediately. 
* **Where to Cut Back Now:** If you are looking to build savings, focus on your flexible spending categories: **Groceries ($300)** and **Shopping ($150)**. Trimming non-essential shopping or meal planning for groceries can free up extra cash each month.


In [22]:

def finance_chat(question, history):
    if not question or not question.strip():
        return "Please enter a question."

    return ask_finance_assistant(question)


chat_app = gr.ChatInterface(
    fn=finance_chat,
    title="Student Budget and Savings Assistant",
    description="Ask questions about your spending and learn how to save money."
)

chat_app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e5a8f6b234ec5e3b3b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:

def combined_finance_app(file_path, question):
    if file_path is None:
        return "Please upload your transactions CSV first."

    try:
        df = pd.read_csv(file_path)

        if "category" not in df.columns or "amount" not in df.columns:
            return "Your CSV must contain category and amount columns."

        df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

        if df["amount"].isna().any() or (df["amount"] < 0).any():
            return "Please check that all amounts are valid, non-negative numbers."

        if df["category"].isna().any() or (df["category"].astype(str).str.strip() == "").any():
            return "Please make sure every transaction has a category."

        transactions = df.to_dict("records")
        spending_summary = analyse_spending(transactions)

        if not question or not question.strip():
            return f"Your spending summary is: {spending_summary}"

        prompt = f"""
        You are a helpful Student Budget and Savings Assistant.

        The user's spending summary is:
        {spending_summary}

        The user asks:
        {question}

        Answer clearly using the supplied spending data.
        Give practical student budgeting advice.
        Do not invent information about income or savings goals.
        """

        return ask_gemini(prompt)

    except (ValueError, TypeError):
        return "There was a problem with the CSV data. Please check your file."

    except Exception:
        return "Something went wrong. Please check your CSV and try again."


combined_app = gr.Interface(
    fn=combined_finance_app,
    inputs=[
        gr.File(
            label="Upload your transactions CSV",
            file_types=[".csv"],
            type="filepath"
        ),
        gr.Textbox(
            label="Ask a finance question",
            placeholder="Which category am I spending the most on?"
        )
    ],
    outputs=gr.Textbox(label="Finance assistant response"),
    title="Student Budget and Savings Assistant",
    description="Upload your transaction data and ask questions about your spending."
)

combined_app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4ad3f9206dc7f184c8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [27]:

import gradio as gr
import pandas as pd
import base64

# Load your Korilakkuma GIF
with open("/content/koriakkuma gif.gif", "rb") as image_file:
    image_data = base64.b64encode(image_file.read()).decode("utf-8")

# Cute pink styling
cute_css = """
.gradio-container {
    background: linear-gradient(135deg, #fff0f6, #ffe4ef, #fff7fb) !important;
    font-family: 'Trebuchet MS', sans-serif !important;
}

.gradio-container h1 {
    color: #d63384 !important;
    text-align: center !important;
    font-weight: 800 !important;
}

.gradio-container h2,
.gradio-container h3,
.gradio-container label {
    color: #a83270 !important;
}

.gradio-container button {
    background: linear-gradient(90deg, #ff85b3, #e879c0) !important;
    color: white !important;
    border: 2px solid #ffc2dc !important;
    border-radius: 22px !important;
    font-weight: bold !important;
}

.gradio-container button:hover {
    transform: scale(1.02);
}

.gradio-container input,
.gradio-container textarea {
    border-radius: 16px !important;
    border-color: #f5a6ca !important;
}

.gradio-container .block {
    border-color: #ffc2dc !important;
    border-radius: 20px !important;
}

footer {
    display: none !important;
}
"""

# Process the uploaded CSV and answer the question
def combined_finance_app(file_path, question):
    if file_path is None:
        return "🎀 Please upload your transactions CSV first!"

    try:
        df = pd.read_csv(file_path)

        if "category" not in df.columns or "amount" not in df.columns:
            return "💗 Your CSV needs 'category' and 'amount' columns."

        df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

        if df["amount"].isna().any() or (df["amount"] < 0).any():
            return "🌸 Please check that all amounts are valid, non-negative numbers."

        if (
            df["category"].isna().any()
            or (df["category"].astype(str).str.strip() == "").any()
        ):
            return "🎀 Please give every transaction a category."

        transactions = df.to_dict("records")
        spending_summary = analyse_spending(transactions)

        if "error" in spending_summary:
            return spending_summary["error"]

        if not question or not question.strip():
            return f"🌷 Your spending summary:\n{spending_summary}"

        prompt = f"""
You are Pink Petals Finance, a friendly student budgeting assistant.

The user's spending summary is:
{spending_summary}

The user asks:
{question}

Answer in a warm, clear and supportive way.
Use the supplied spending data.
Give practical budgeting suggestions.
Do not invent income, savings goals or other financial information
that the user has not provided.
"""

        return ask_gemini(prompt)

    except (ValueError, TypeError):
        return "💗 There was a problem with the CSV data. Please check your file."

    except Exception:
        return "🌸 Something went wrong. Please try again."


# Build the cute app
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="pink",
        secondary_hue="rose",
        neutral_hue="pink"
    ),
    css=cute_css
) as combined_app:

    # Korilakkuma GIF decoration
    gr.HTML(f"""
    <div style="text-align: center; padding: 12px;">
        <img
            src="data:image/gif;base64,{image_data}"
            alt="Korilakkuma cheerleader"
            style="width: 180px; max-width: 70%; height: auto;"
        />
        <p style="color: #d63384; font-weight: bold; font-size: 16px;">
            ♡ ૮₍ ˶ᵔ ᵕ ᵔ˶ ₎ა  ♡
        </p>
    </div>
    """)

    gr.Markdown("""
    # 🎀 ♡ Deekshi's Finance Anaylser! <3 ;3 ♡ 🎀

    ### ✨ Your money, your goals, your glow-up ✨

    Welcome to your little budgeting corner! 🌷

    Upload your spending CSV and ask questions to get
    personalised advice from your finance assistant.

    ˚₊‧꒰ა ♡ ໒꒱ ‧₊˚
    """)

    with gr.Row():
        with gr.Column(scale=1):
            csv_input = gr.File(
                label="🎀 Upload your transactions",
                file_types=[".csv"],
                type="filepath"
            )

            question_input = gr.Textbox(
                label="💌 Ask your finance assistant",
                placeholder="How can I save more money? ♡",
                lines=3
            )

            submit_button = gr.Button("💗 Analyse my spending ✨")

        with gr.Column(scale=1):
            answer_output = gr.Textbox(
                label="🌷 Your personalised money advice",
                lines=10,
                interactive=False
            )

    gr.Markdown("""
    ---

    ### ♡ Little reminders ♡

    🌸 Every dollar counts.

    🎀 Small savings add up.

    ✨ You're doing amazing!

    <div style="text-align:center; color:#d63384;">
        ♡
    </div>
    """)

    submit_button.click(
        fn=combined_finance_app,
        inputs=[csv_input, question_input],
        outputs=answer_output
    )

# Launch the app
combined_app.launch()


/tmp/ipykernel_1102/1018453992.py:113: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://09d28165675051002b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
ISYS2001 Developer's Diary
Project: Student Budget and Savings Assistant
Student: Deekshitha Kumarasamy
Week: 19–25 October 2026

Entry 1  Planning the project
Date: 19 October 2026
Goal: Identify a problem to solve and plan the finance assistant.
AI prompt used: Help me develop a student budget and savings assistant for my business programming assignment.
AI output: The proposed solution was a Python application that helps students understand their spending by analysing transaction data, calculating expenses by category and providing budgeting advice.
My decision: I accepted the idea because students can use it to understand their spending habits.
What I learned: Planning the problem, inputs, processing and outputs before coding helps organise the development process.

Entry 2  Developing the basic Python calculations
Date: 20 October 2026
Goal: Create a function that calculates how much money remains after expenses.
AI prompt used: Help me write a simple Python function to calculate the remaining money after expenses.
AI output: The AI provided a function called calculate_remaining() that adds the expenses and subtracts the total from the income.
My decision: I accepted the function and tested it using an income of $1,800 and expenses of $700, $250, $100 and $150. The result was $600 remaining.
What I learned: Functions allow calculations to be reused, while lists and the built-in sum() function make it easier to process multiple expenses.

Entry 3  Creating the spending analysis tool
Date: 20 October 2026
Goal: Analyse spending by category.
AI prompt used: Help me create a Python function that groups transaction amounts by category using a dictionary.
AI output: The AI suggested an analyse_spending() function that loops through transaction records and adds amounts belonging to the same category. It also included error handling for invalid transaction data.
My decision: I accepted the approach and tested it using sample transactions. The function grouped the two grocery transactions into a total of $300 and returned totals for Rent, Groceries, Transport and Shopping.
What I learned: Dictionaries can store category totals, loops process each transaction, conditionals validate amounts, and try/except blocks handle certain errors.

Entry 4  Working with CSV files and pandas
Date: 21 October 2026
Goal: Allow the application to analyse transaction data stored in a CSV file.
AI prompt used: Help me create and load a transaction CSV using pandas, then connect it to my spending analysis function.
AI output: The AI provided code using pandas to create a sample DataFrame, save it as transactions.csv, read the file and convert its rows into dictionaries.
My decision: I accepted the approach and ran the cells in Google Colab. When a function was temporarily unavailable because the cells had not been executed in the correct order, I reran the relevant cells and confirmed the analysis worked.
What I learned: Pandas makes it easier to load and process tabular data. In a notebook, cells defining functions must be executed before other cells can use those functions.

Entry 5  Adding CSV validation and a Gradio interface
Date: 21 October 2026
Goal: Allow users to upload transaction files and receive a spending analysis.
AI prompt used: Help me create a Gradio interface where users can upload a CSV containing category and amount columns.
AI output: The AI provided an upload-handling function and a Gradio interface. The code checks for required columns, converts amounts to numeric values and checks for invalid or negative amounts and missing categories.
My decision: I accepted the interface and tested the relevant CSV and spending analysis cells. I then extended the application to accept a user's question alongside their uploaded CSV.
What I learned: Gradio can connect Python functions to interface components, allowing users to interact with a program without running each function manually.

Entry 6  Connecting the Gemini API
Date: 22 October 2026
Goal: Enable the assistant to answer budgeting questions using AI.
AI prompt used: Help me connect my Python finance assistant to the Gemini API using requests.
AI output: The AI provided a function that sends a question to the Gemini API, reads the response and handles connection errors and unexpected response formats.
My decision: I accepted the approach and tested it with a question asking for a student money-saving tip. The API connection worked. I then created a further function that passes the spending summary and user's question to Gemini.
What I learned: APIs allow Python programs to communicate with external services. The requests library sends HTTP requests, and the returned JSON data can be processed to retrieve the answer.
Security consideration: I stored the API key in Google Colab Secrets so it would not need to be written directly into the application code.

Entry 7  Combining the application
Date: 22 October 2026
Goal: Combine the CSV upload and AI question-answering features into one interface.
AI prompt used: Help me combine the CSV upload and finance chat into a single Gradio application.
AI output: The AI provided a combined function that reads an uploaded CSV, validates the transaction data, generates a spending summary and sends the user's question and summary to Gemini.
My decision: I accepted the combined approach because it allows users to upload their data and ask questions in one place. I retained the existing spending analysis and Gemini functions.
What I learned: Separate functions can be combined into a larger application. Validating user input before processing it helps prevent invalid data from being used in calculations.

Entry 8  Improving the visual design
Date: 22 October 2026
Goal: Make the application more attractive and user-friendly.
AI prompt used: Make my Gradio finance app pink and cute, with decorations, and add my Korilakkuma cheerleader image.
AI output: The AI provided custom CSS styling with pastel pink backgrounds, rounded components, pink buttons, decorative text and an image displayed above the application title.
My decision: I accepted the visual changes because they matched my preferred aesthetic. I kept the finance processing functions separate from the visual styling so that the appearance could be changed without rewriting the main calculations.
What I learned: CSS can customise the appearance of a Gradio interface. HTML can be used to display images and decorative content, while the Python functions continue to handle the application logic.

Weekly reflection
During this week, I developed this Finance app from an initial idea into a working prototype. I started by planning the problem and writing basic Python calculations. I then added spending analysis, CSV processing, validation and a Gradio interface. After that, I connected the Gemini API and customised the app with a pastel pink theme and a Korilakkuma decoration.
AI assistance helped me develop and troubleshoot code, but I also needed to run the cells, check the outputs and correct problems such as functions being called before their definitions had been executed.
I learned how functions, lists, dictionaries, loops, conditionals and error handling can work together in a practical program. I also gained experience using pandas to process CSV files, requests to call an API and Gradio to build a user interface.
The project could be improved further by adding income and savings-goal inputs, calculating remaining income, adding more automated tests and making the chat maintain conversation history.


In [31]:

# ISYS2001 Assertion Tests

# Test 1: Normal expense calculation
assert calculate_remaining(1800, [700, 250, 100, 150]) == 600

# Test 2: No expenses
assert calculate_remaining(1000, []) == 1000

# Test 3: Expenses equal income
assert calculate_remaining(1000, [400, 600]) == 0

# Test 4: Expenses exceed income
assert calculate_remaining(500, [300, 300]) == -100

# Test 5: Spending grouped by category
sample_transactions = [
    {"category": "Rent", "amount": 700},
    {"category": "Groceries", "amount": 250},
    {"category": "Transport", "amount": 100},
    {"category": "Groceries", "amount": 50},
    {"category": "Shopping", "amount": 150}
]

expected_result = {
    "Rent": 700,
    "Groceries": 300,
    "Transport": 100,
    "Shopping": 150
}

assert analyse_spending(sample_transactions) == expected_result

# Test 6: Negative expense amount is rejected
assert analyse_spending([
    {"category": "Shopping", "amount": -50}
]) == {"error": "Expense amounts must be non-negative numbers."}

print("All six assertion tests passed!")


All six assertion tests passed!
